# BTC Hourly Technical Indicators

Extended version of `OHLCV-hourly.ipynb` with comprehensive technical indicators.

**Categories**: Order Flow · Returns · Momentum · Trend · Volatility · Volume · Multi-Timeframe

**Note**: Dec 2017 is downloaded as warmup to avoid NaN at the 2018-01-01 boundary (longest lookback = SMA-200 = 200 candles; Dec 2017 provides ~744 candles).

In [1]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io

# --- 1. CONFIGURATION ---
symbol = 'BTCUSDT'
interval = '1h'
base_url = "https://data.binance.vision/data/spot/monthly/klines"
all_frames = []
EPS = 1e-10  # Epsilon to prevent division-by-zero

print(f"\U0001f4e1 Downloading {symbol} Hourly Data (Dec 2017 - Dec 2025)...")

# --- 2. DOWNLOAD LOOP ---
# Dec 2017 is fetched to "warm up" rolling windows (SMA-200 needs 200 candles;
# Dec 2017 provides ~744 candles, so every indicator is fully valid by Jan 1 2018).
fetch_list = [(2017, 12)] + [(y, m) for y in range(2018, 2026) for m in range(1, 13)]

for year, month in fetch_list:
    m_str = f"{month:02d}"
    file_name = f"{symbol}-{interval}-{year}-{m_str}.zip"
    url = f"{base_url}/{symbol}/{interval}/{file_name}"
    
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                csv_name = file_name.replace('.zip', '.csv')
                with z.open(csv_name) as f:
                    all_frames.append(pd.read_csv(f, header=None))
            print(f"\u2705 Loaded {year}-{m_str}")
        else:
            print(f"\u26a0\ufe0f Skipped {year}-{m_str}: Status {response.status_code}")
    except requests.exceptions.RequestException as e:
        print(f"\u274c Failed to download {year}-{m_str}: {e}")

📡 Downloading BTCUSDT Hourly Data (Dec 2017 - Dec 2025)...
✅ Loaded 2017-12
✅ Loaded 2018-01
✅ Loaded 2018-02
✅ Loaded 2018-03
✅ Loaded 2018-04
✅ Loaded 2018-05
✅ Loaded 2018-06
✅ Loaded 2018-07
✅ Loaded 2018-08
✅ Loaded 2018-09
✅ Loaded 2018-10
✅ Loaded 2018-11
✅ Loaded 2018-12
✅ Loaded 2019-01
✅ Loaded 2019-02
✅ Loaded 2019-03
✅ Loaded 2019-04
✅ Loaded 2019-05
✅ Loaded 2019-06
✅ Loaded 2019-07
✅ Loaded 2019-08
✅ Loaded 2019-09
✅ Loaded 2019-10
✅ Loaded 2019-11
✅ Loaded 2019-12
✅ Loaded 2020-01
✅ Loaded 2020-02
✅ Loaded 2020-03
✅ Loaded 2020-04
✅ Loaded 2020-05
✅ Loaded 2020-06
✅ Loaded 2020-07
✅ Loaded 2020-08
✅ Loaded 2020-09
✅ Loaded 2020-10
✅ Loaded 2020-11
✅ Loaded 2020-12
✅ Loaded 2021-01
✅ Loaded 2021-02
✅ Loaded 2021-03
✅ Loaded 2021-04
✅ Loaded 2021-05
✅ Loaded 2021-06
✅ Loaded 2021-07
✅ Loaded 2021-08
✅ Loaded 2021-09
✅ Loaded 2021-10
✅ Loaded 2021-11
✅ Loaded 2021-12
✅ Loaded 2022-01
✅ Loaded 2022-02
✅ Loaded 2022-03
✅ Loaded 2022-04
✅ Loaded 2022-05
✅ Loaded 2022-06
✅ Load

In [2]:
# --- 3. DYNAMIC TIMESTAMP CONVERSION ---
# Binance switched from millisecond to microsecond timestamps around mid-2024.
# This function handles both formats transparently.

if all_frames:
    df = pd.concat(all_frames, ignore_index=True)
    df.columns = ['open_time', 'open', 'high', 'low', 'close', 'volume',
                  'close_time', 'q_vol', 'count', 'tb_base', 'tb_quote', 'ignore']

    def convert_binance_time(ts):
        """Handle Binance's mid-2024 shift from millisecond to microsecond timestamps."""
        if ts > 10**14:
            return pd.to_datetime(ts, unit='us', utc=True)  # Microseconds
        return pd.to_datetime(ts, unit='ms', utc=True)      # Milliseconds

    print("\u23f3 Converting timestamps to strictly formatted UTC...")
    df['timestamp'] = df['open_time'].apply(convert_binance_time)

    df = df.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    df = df[['timestamp', 'open', 'high', 'low', 'close', 'volume',
             'q_vol', 'count', 'tb_base']].copy()

    print(f"\u2705 Timestamp conversion complete. Total raw rows: {len(df)}")
else:
    raise RuntimeError("No data was downloaded. Check your network connection.")

⏳ Converting timestamps to strictly formatted UTC...
✅ Timestamp conversion complete. Total raw rows: 70752


## Feature Engineering — Technical Indicators

All features are computed on the full dataset (including Dec 2017 warmup).
NaN rows from rolling window warm-up are dropped later, before filtering to 2018+.

In [3]:
# ═══════════════════════════════════════════════════════════════
# 4.1 ORDER FLOW FEATURES (6 features)
# ═══════════════════════════════════════════════════════════════
print("\U0001f4ca Computing Order Flow features...")

df['taker_sell_vol']       = df['volume'] - df['tb_base']
df['volume_delta']         = df['tb_base'] - df['taker_sell_vol']
df['avg_trade_size']       = np.where(df['count'] > 0, df['volume'] / df['count'], 0)
df['vol_delta_sma_24']     = df['volume_delta'].rolling(window=24).mean()
df['avg_trade_size_sma_24']= df['avg_trade_size'].rolling(window=24).mean()
df['vol_delta_ema_24']     = df['volume_delta'].ewm(span=24, adjust=False).mean()

print("   \u2705 Order Flow: taker_sell_vol, volume_delta, avg_trade_size, vol_delta_sma_24, avg_trade_size_sma_24, vol_delta_ema_24")

📊 Computing Order Flow features...
   ✅ Order Flow: taker_sell_vol, volume_delta, avg_trade_size, vol_delta_sma_24, avg_trade_size_sma_24, vol_delta_ema_24


In [4]:
# ═══════════════════════════════════════════════════════════════
# 4.2 RETURNS & LOG RETURNS (6 features)
# ═══════════════════════════════════════════════════════════════
print("\U0001f4ca Computing Returns...")

for shift, label in [(1, '1h'), (4, '4h'), (12, '12h')]:
    prev = df['close'].shift(shift)
    df[f'return_{label}']     = (df['close'] - prev) / (prev + EPS)
    df[f'log_return_{label}'] = np.log(df['close'] / (prev + EPS))

print("   \u2705 Returns: return_1h, return_4h, return_12h, log_return_1h, log_return_4h, log_return_12h")

📊 Computing Returns...
   ✅ Returns: return_1h, return_4h, return_12h, log_return_1h, log_return_4h, log_return_12h


In [5]:
# ═══════════════════════════════════════════════════════════════
# 4.3 MOMENTUM INDICATORS (9 features)
# ═══════════════════════════════════════════════════════════════
print("\U0001f4ca Computing Momentum indicators...")

# --- RSI (Wilder's method, 3 periods) ---
delta = df['close'].diff()
gain = (delta.where(delta > 0, 0)).fillna(0)
loss = (-delta.where(delta < 0, 0)).fillna(0)

for period in [7, 14, 28]:
    avg_gain = gain.ewm(com=period-1, adjust=False).mean()
    avg_loss = loss.ewm(com=period-1, adjust=False).mean()
    rs = avg_gain / (avg_loss + EPS)
    df[f'RSI_{period}'] = 100 - (100 / (1 + rs))

# --- Stochastic Oscillator (14-period %K, 3-period %D) ---
lowest_14  = df['low'].rolling(14).min()
highest_14 = df['high'].rolling(14).max()
df['stoch_k'] = 100 * (df['close'] - lowest_14) / (highest_14 - lowest_14 + EPS)
df['stoch_d'] = df['stoch_k'].rolling(3).mean()

# --- Williams %R (14-period) ---
df['williams_r'] = -100 * (highest_14 - df['close']) / (highest_14 - lowest_14 + EPS)

# --- Rate of Change (12 & 24 period) ---
df['roc_12'] = (df['close'] - df['close'].shift(12)) / (df['close'].shift(12) + EPS) * 100
df['roc_24'] = (df['close'] - df['close'].shift(24)) / (df['close'].shift(24) + EPS) * 100

# --- Commodity Channel Index (20-period) ---
tp = (df['high'] + df['low'] + df['close']) / 3
tp_sma = tp.rolling(20).mean()
tp_mad = tp.rolling(20).apply(lambda x: np.abs(x - x.mean()).mean(), raw=True)
df['cci_20'] = (tp - tp_sma) / (0.015 * tp_mad + EPS)

print("   \u2705 Momentum: RSI_7, RSI_14, RSI_28, stoch_k, stoch_d, williams_r, roc_12, roc_24, cci_20")

📊 Computing Momentum indicators...
   ✅ Momentum: RSI_7, RSI_14, RSI_28, stoch_k, stoch_d, williams_r, roc_12, roc_24, cci_20


In [6]:
# ═══════════════════════════════════════════════════════════════
# 4.4 TREND INDICATORS (9 features)
# ═══════════════════════════════════════════════════════════════
print("\U0001f4ca Computing Trend indicators...")

# EMAs
df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()
df['ema_26'] = df['close'].ewm(span=26, adjust=False).mean()

# SMAs
df['sma_50']  = df['close'].rolling(50).mean()
df['sma_200'] = df['close'].rolling(200).mean()

# MACD
df['MACD_line']      = df['ema_12'] - df['ema_26']
df['MACD_signal']    = df['MACD_line'].ewm(span=9, adjust=False).mean()
df['MACD_histogram'] = df['MACD_line'] - df['MACD_signal']

# Crossover signals (binary)
df['ema_12_26_cross']     = (df['ema_12'] > df['ema_26']).astype(int)
df['price_above_sma_200'] = (df['close'] > df['sma_200']).astype(int)

print("   \u2705 Trend: ema_12, ema_26, sma_50, sma_200, MACD_line, MACD_signal, MACD_histogram, ema_12_26_cross, price_above_sma_200")

📊 Computing Trend indicators...
   ✅ Trend: ema_12, ema_26, sma_50, sma_200, MACD_line, MACD_signal, MACD_histogram, ema_12_26_cross, price_above_sma_200


In [7]:
# ═══════════════════════════════════════════════════════════════
# 4.5 VOLATILITY INDICATORS (8 features)
# ═══════════════════════════════════════════════════════════════
print("\U0001f4ca Computing Volatility indicators...")

# True Range
high_low   = df['high'] - df['low']
high_close = np.abs(df['high'] - df['close'].shift(1))
low_close  = np.abs(df['low'] - df['close'].shift(1))
tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)

# ATR
df['ATR_14'] = tr.rolling(window=14).mean()
df['ATR_7']  = tr.rolling(window=7).mean()

# Bollinger Bands (20-period, 2 sigma)
sma_20 = df['close'].rolling(20).mean()
std_20 = df['close'].rolling(20).std()
df['bb_upper'] = sma_20 + 2 * std_20
df['bb_lower'] = sma_20 - 2 * std_20
df['bb_mid']   = sma_20
df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / (df['bb_mid'] + EPS)
df['bb_pct']   = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + EPS)

# Realized Volatility (rolling std of log returns, 24h)
log_ret = np.log(df['close'] / (df['close'].shift(1) + EPS))
df['realized_vol_24'] = log_ret.rolling(24).std()

print("   \u2705 Volatility: ATR_14, ATR_7, bb_upper, bb_lower, bb_mid, bb_width, bb_pct, realized_vol_24")

📊 Computing Volatility indicators...
   ✅ Volatility: ATR_14, ATR_7, bb_upper, bb_lower, bb_mid, bb_width, bb_pct, realized_vol_24


In [8]:
# ═══════════════════════════════════════════════════════════════
# 4.6 VOLUME INDICATORS (8 features)
# ═══════════════════════════════════════════════════════════════
print("\U0001f4ca Computing Volume indicators...")

# Volume SMA & relative ratio
df['volume_sma_24'] = df['volume'].rolling(24).mean()
df['volume_ratio']  = df['volume'] / (df['volume_sma_24'] + EPS)

# (volume_delta, vol_delta_sma_24, vol_delta_ema_24 already computed in Order Flow)

# On-Balance Volume (OBV) & its EMA
direction = np.sign(df['close'].diff()).fillna(0)
df['obv']        = (df['volume'] * direction).cumsum()
df['obv_ema_24'] = df['obv'].ewm(span=24, adjust=False).mean()

# VWAP ratio (close relative to volume-weighted avg price)
vwap = df['q_vol'] / (df['volume'] + EPS)
df['vwap_ratio'] = df['close'] / (vwap + EPS)

print("   \u2705 Volume: volume_sma_24, volume_ratio, obv, obv_ema_24, vwap_ratio")

📊 Computing Volume indicators...
   ✅ Volume: volume_sma_24, volume_ratio, obv, obv_ema_24, vwap_ratio


In [9]:
# ═══════════════════════════════════════════════════════════════
# 4.7 MULTI-TIMEFRAME FEATURES (5 features)
# ═══════════════════════════════════════════════════════════════
print("\U0001f4ca Computing Multi-Timeframe features...")

df['close_4h_sma']  = df['close'].rolling(4).mean()
df['volume_4h_sum'] = df['volume'].rolling(4).sum()
df['high_12h']      = df['high'].rolling(12).max()
df['low_12h']       = df['low'].rolling(12).min()
df['range_12h_pct'] = (df['high_12h'] - df['low_12h']) / (df['close'] + EPS)

print("   \u2705 Multi-TF: close_4h_sma, volume_4h_sum, high_12h, low_12h, range_12h_pct")

📊 Computing Multi-Timeframe features...
   ✅ Multi-TF: close_4h_sma, volume_4h_sum, high_12h, low_12h, range_12h_pct


In [10]:
# --- 5. FILTER & EXPORT ---
print("\u2702\ufe0f Filtering data to start exactly from 2018-01-01 00:00:00 UTC...")
df = df.dropna().reset_index(drop=True)
df = df[df['timestamp'] >= '2018-01-01'].reset_index(drop=True)

# Add Trend Label
df['trend'] = np.where(df['close'] > df['open'], 'Bullish', 'Bearish')

# Export
df.to_csv('btc_hourly_indicators.csv', index=False)

print(f"\u2728 Success! Total Rows: {len(df)}")
print(f"Data starts at: {df['timestamp'].min()}")
print(f"Data ends at: {df['timestamp'].max()}")
print(f"Total features: {len(df.columns)}")
print(f"Features: {list(df.columns)}")

✂️ Filtering data to start exactly from 2018-01-01 00:00:00 UTC...
✨ Success! Total Rows: 70008
Data starts at: 2018-01-01 00:00:00+00:00
Data ends at: 2025-12-31 23:00:00+00:00
Total features: 58
Features: ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'q_vol', 'count', 'tb_base', 'taker_sell_vol', 'volume_delta', 'avg_trade_size', 'vol_delta_sma_24', 'avg_trade_size_sma_24', 'vol_delta_ema_24', 'return_1h', 'log_return_1h', 'return_4h', 'log_return_4h', 'return_12h', 'log_return_12h', 'RSI_7', 'RSI_14', 'RSI_28', 'stoch_k', 'stoch_d', 'williams_r', 'roc_12', 'roc_24', 'cci_20', 'ema_12', 'ema_26', 'sma_50', 'sma_200', 'MACD_line', 'MACD_signal', 'MACD_histogram', 'ema_12_26_cross', 'price_above_sma_200', 'ATR_14', 'ATR_7', 'bb_upper', 'bb_lower', 'bb_mid', 'bb_width', 'bb_pct', 'realized_vol_24', 'volume_sma_24', 'volume_ratio', 'obv', 'obv_ema_24', 'vwap_ratio', 'close_4h_sma', 'volume_4h_sum', 'high_12h', 'low_12h', 'range_12h_pct', 'trend']


In [11]:
# --- 6. VERIFICATION ---
print("\n" + "=" * 60)
print("  VERIFICATION REPORT")
print("=" * 60)

trend_counts = df['trend'].value_counts()
print(f"\nTrend Distribution:")
print(trend_counts)

print(f"\nNaN Check: {df.isna().sum().sum()} remaining NaNs")

# Sanity checks
print(f"\nSanity Checks:")
for col in ['RSI_7', 'RSI_14', 'RSI_28']:
    print(f"  {col}: [{df[col].min():.2f}, {df[col].max():.2f}] (expected 0-100)")
for col in ['ATR_14', 'ATR_7']:
    print(f"  {col}: min={df[col].min():.4f} (expected >= 0)")
print(f"  bb_pct: [{df['bb_pct'].min():.4f}, {df['bb_pct'].max():.4f}]")

print(f"\n\U0001f4ca Quick Stats:")
print(df.describe().round(4).to_string())


  VERIFICATION REPORT

Trend Distribution:
trend
Bullish    35580
Bearish    34428
Name: count, dtype: int64

NaN Check: 0 remaining NaNs

Sanity Checks:
  RSI_7: [1.45, 99.29] (expected 0-100)
  RSI_14: [4.37, 96.41] (expected 0-100)
  RSI_28: [10.03, 91.28] (expected 0-100)
  ATR_14: min=8.0543 (expected >= 0)
  ATR_7: min=7.0943 (expected >= 0)
  bb_pct: [-0.5486, 1.5479]

📊 Quick Stats:
              open         high          low        close       volume         q_vol         count     tb_base  taker_sell_vol  volume_delta  avg_trade_size  vol_delta_sma_24  avg_trade_size_sma_24  vol_delta_ema_24   return_1h  log_return_1h   return_4h  log_return_4h  return_12h  log_return_12h       RSI_7      RSI_14      RSI_28     stoch_k     stoch_d  williams_r      roc_12      roc_24      cci_20       ema_12       ema_26       sma_50      sma_200   MACD_line  MACD_signal  MACD_histogram  ema_12_26_cross  price_above_sma_200      ATR_14       ATR_7     bb_upper     bb_lower       bb_mid    bb